# Frozen features

In [ ]:
import sys
from pathlib import Path

project_dir = Path("/kaggle/input/datasets/denysmykhailov/alseg-source/source")
sys.path.insert(0, str(project_dir))


In [ ]:
%pip install -q --no-cache-dir uv
!uv pip install --python "{sys.executable}" -r "{project_dir}/pyproject.toml" --extra encoder --cache-dir /tmp/alseg-uv


In [ ]:
from alseg.config import EncoderConfig, EncoderName, ExperimentConfig, get_project_version

config = ExperimentConfig(
    encoder=EncoderConfig(image_size=448, batch_size=8),
    paths={"cache_dir": "/tmp/alseg-weights"},
    tracking={"mode": "disabled"},
)
encoder_0 = EncoderName.DINO_VITB16
encoder_1 = EncoderName.IBOT_VITB16
output_dir = Path("/kaggle/working/alseg-features")
config_path = config.save("/tmp/alseg-extraction.json")
extract_script = project_dir / "script" / "extract_features.py"
print("Source version:", get_project_version())
print("GPU 0:", encoder_0, "| GPU 1:", encoder_1)
config.show()


In [ ]:
%run "$extract_script" --config "$config_path" --encoder "$encoder_0" --encoder "$encoder_1" --gpus 2 --output-dir "$output_dir"


In [ ]:
import json

summary = json.loads((output_dir / "extraction.json").read_text())
assert summary["complete"] and summary["reusable"], "Extraction did not finish a full cache"
for name in summary["encoders"]:
    report = json.loads((output_dir / name / "metadata.json").read_text())
    print(name, report["device"], report["gpu"])
    print("Train:", report["splits"]["train"]["patches_shape"])
    print("Val:  ", report["splits"]["val"]["patches_shape"])
    print("Seconds:", round(report["extraction_seconds"], 1))
print("Total wall seconds:", round(summary["wall_seconds"], 1))
print("Ready to share:", output_dir)


Example usage after:

```python
import numpy as np
import torch
from alseg.config import EncoderConfig, EncoderName, ExperimentConfig
from alseg.nn.encoders import load_encoder

features_root = Path("/kaggle/input/notebooks/denysmykhailov/active-learning-gpu/alseg-features")
name = EncoderName.DINO_VITB16  # Or EncoderName.IBOT_VITB16.
config = ExperimentConfig(
    encoder=EncoderConfig(name=name, image_size=448, features_dir=features_root / name),
)
encoder = load_encoder(config.encoder.model_dump(mode="json"), config.paths.cache_dir, config.device)
embeddings = torch.from_numpy(np.load(features_root / name / "train" / "embeddings.npy", allow_pickle=False))
# Pass encoder, embeddings and config.model_dump(mode="json") to run_active_learning.
```

`iter_features` reads the requested rows from disk; it does not decode RGB or run a backbone in cache mode.
Original VOC masks still come only through Oracle or official evaluation access.
